In [2]:
import pandas as pd
import numpy as np
import math
import os

# ==============================================================
# 1) PATHS AND PARAMETERS
# ==============================================================
INPUT_DIR = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files"
OUTPUT_DIR = r"C:\Users\loci_\Desktop\trading_webapp\DATA"

RX_FILE = os.path.join(INPUT_DIR, "RX1_small.csv")
AD_FILE = os.path.join(INPUT_DIR, "AD1_small.csv")

NAV_START = 10_000_000
TARGET_VOL_ANNUAL = 0.20
TRADING_DAYS = 256
LOOKBACK_FOR_VAR = 36
DECAY = 2 / (LOOKBACK_FOR_VAR + 1)

# portfolio weights (must sum to 1)
WEIGHTS = {"RX1": 0.5, "AD1": 0.5}

# ==============================================================
# 2) HELPERS
# ==============================================================
def calc_sharpe(series, annualisation=256):
    s = series.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) <= 1 or s.std() == 0:
        return np.nan
    return s.mean() / s.std() * np.sqrt(annualisation)

def compute_turnover(df, trading_days_per_year=256):
    if "trades" not in df.columns or "current_position" not in df.columns:
        return np.nan, np.nan, np.nan
    years = len(df) / trading_days_per_year
    avg_lots_traded_yearly = df["trades"].abs().sum() / years
    avg_abs_position = df["current_position"].abs().mean()
    if avg_abs_position == 0:
        turnover = np.nan
    else:
        turnover = avg_lots_traded_yearly / (2 * avg_abs_position)
    return turnover, avg_lots_traded_yearly, avg_abs_position

# ==============================================================
# 3) INSTRUMENT PIPELINE
# ==============================================================
def run_instrument(df, instr_name, weight, pdm, output_dir):
    """
    df must have at least:
      - Date
      - PX_CLOSE_1D
      - st_dev
      - Exchange rate
      - (optional) TICK_VALUE, TICK_SIZE, POINT_VALUE
    """
    df = df.copy()
    df = df.sort_values("Date").set_index("Date")

    price = df["PX_CLOSE_1D"].astype(float)
    stdev_price = df["st_dev"].astype(float)

    # FX: in your file it's >1 → EUR→USD, so we MULTIPLY
    if "Exchange rate" in df.columns:
        fx_series = df["Exchange rate"].astype(float)
    else:
        fx_series = pd.Series(1.0, index=df.index)

    # tick / point info
    tick_value = float(df["TICK_VALUE"].iloc[0]) if "TICK_VALUE" in df.columns else 10.0
    tick_size  = float(df["TICK_SIZE"].iloc[0]) if "TICK_SIZE" in df.columns else 0.01
    mult = tick_value / tick_size
    point_value = float(df["POINT_VALUE"].iloc[0]) if "POINT_VALUE" in df.columns else mult

    # 1) returns
    df["ret_pct_for_sig"] = price.pct_change()
    df["ret_net_for_var"] = price.diff()

    # 2) EWMA variance (your formula)
    variance = [np.nan] * len(df)
    ret_net_vals = df["ret_net_for_var"].values
    # first obs = squared return
    first_idx = np.where(~np.isnan(ret_net_vals))[0][0]
    first_sq = ret_net_vals[first_idx] ** 2
    variance[first_idx] = first_sq
    prev_var = first_sq
    for i in range(first_idx + 1, len(df)):
        rn = ret_net_vals[i]
        rn2 = 0.0 if np.isnan(rn) else rn * rn
        new_var = DECAY * rn2 + (1 - DECAY) * prev_var
        variance[i] = new_var
        prev_var = new_var
    df["variance"] = variance
    df["stdev_variance"] = np.sqrt(df["variance"])

    # 3) signal: 4/16
    df["ewma_fast"] = price.ewm(span=4, adjust=False).mean()
    df["ewma_slow"] = price.ewm(span=16, adjust=False).mean()
    df["raw_crossover"] = df["ewma_fast"] - df["ewma_slow"]
    df["vol_adj_crossover"] = df["raw_crossover"] / df["stdev_variance"]
    df["scaled_forecast"] = df["vol_adj_crossover"] * 7.5
    df["capped_forecast"] = df["scaled_forecast"].clip(-20, 20)

    # signal PnL (raw layer)
    df["forecast_x_return"] = df["capped_forecast"].shift(1) * df["ret_pct_for_sig"]

    # 4) modular / cash vol chain
    df["price_volatility"] = (stdev_price / price * 100).round(2)
    df["one_pct_move"] = price * 0.01
    df["block_value_eur"] = df["one_pct_move"] * point_value
    df["icv_eur"] = df["price_volatility"] * df["block_value_eur"]
    # ✅ FX column is >1 → multiply EUR→USD
    df["ivv"] = df["icv_eur"] * fx_series

    # 5) iterative sizing
    nav_list = []
    daily_cash_vol_list = []
    target_contracts_list = []
    trades_list = []
    pnl_usd_list = []
    carry_pnl_usd_list = []
    trade_pnl_usd_list = []
    strategy_ret_list = []
    vol_scaler_list = []
    subsystem_pos_list = []
    port_instr_pos_list = []

    prev_nav = NAV_START
    prev_daily_cash_vol = prev_nav * TARGET_VOL_ANNUAL / np.sqrt(TRADING_DAYS)
    prev_target_contracts = 0

    for i, (dt, row) in enumerate(df.iterrows()):
        px_today = row["PX_CLOSE_1D"]

        # 1) today's risk budget = yesterday's NAV
        daily_cash_vol_target = prev_daily_cash_vol

        # 2) vol-scaler
        ivv_today = row["ivv"]
        if np.isnan(ivv_today) or ivv_today == 0:
            vol_scaler_today = np.nan
        else:
            vol_scaler_today = daily_cash_vol_target / ivv_today

        # 3) subsystem position
        cap_f = row["capped_forecast"]
        if np.isnan(cap_f) or np.isnan(vol_scaler_today):
            subsystem_pos = 0.0
        else:
            subsystem_pos = (cap_f * vol_scaler_today) / 10.0

        # 4) portfolio instrument position (before rounding)
        port_instr_pos = subsystem_pos * weight * pdm

        # 5) round to contracts
        if np.isnan(port_instr_pos):
            target_contracts = 0
        else:
            target_contracts = int(round(port_instr_pos))

        # 6) trades
        trades = target_contracts - prev_target_contracts

        # 7) PnL
        if i == 0:
            delta_price = 0.0
        else:
            prev_price = df["PX_CLOSE_1D"].iloc[i - 1]
            delta_price = px_today - prev_price

        carry_pnl_eur = prev_target_contracts * delta_price * mult
        fx_today = fx_series.loc[dt]
        carry_pnl_usd = carry_pnl_eur * fx_today

        # placeholder exec PnL (we don't have actual exec prices)
        trade_pnl_usd = 0.0

        pnl_usd = carry_pnl_usd + trade_pnl_usd
        nav_today = prev_nav + pnl_usd
        strat_ret = pnl_usd / prev_nav if prev_nav != 0 else 0.0

        # store
        nav_list.append(nav_today)
        daily_cash_vol_list.append(daily_cash_vol_target)
        target_contracts_list.append(target_contracts)
        trades_list.append(trades)
        pnl_usd_list.append(pnl_usd)
        carry_pnl_usd_list.append(carry_pnl_usd)
        trade_pnl_usd_list.append(trade_pnl_usd)
        strategy_ret_list.append(strat_ret)
        vol_scaler_list.append(vol_scaler_today)
        subsystem_pos_list.append(subsystem_pos)
        port_instr_pos_list.append(port_instr_pos)

        # roll
        prev_nav = nav_today
        prev_daily_cash_vol = nav_today * TARGET_VOL_ANNUAL / np.sqrt(TRADING_DAYS)
        prev_target_contracts = target_contracts

    # assign back
    df["nav_usd"] = nav_list
    df["daily_cash_vol_target"] = daily_cash_vol_list
    df["volatility_scaler"] = vol_scaler_list
    df["subsystem_position"] = subsystem_pos_list
    df["portfolio_instr_position"] = port_instr_pos_list
    df["target_contracts"] = target_contracts_list
    df["trades"] = trades_list
    df["pnl_usd"] = pnl_usd_list
    df["carry_pnl_usd"] = carry_pnl_usd_list
    df["trade_pnl_usd"] = trade_pnl_usd_list
    df["strategy_ret"] = strategy_ret_list
    df["current_position"] = df["target_contracts"].shift(1).fillna(0)
    df["cumulative_pnl_usd"] = df["nav_usd"] - NAV_START
    df["pdm_used"] = pdm
    df["eff_multiplier"] = weight * pdm

    # save instrument output
    out_file = os.path.join(OUTPUT_DIR, f"{instr_name}_full_chain.csv")
    df.reset_index().to_csv(out_file, index=False)
    print(f"✅ Saved {instr_name} to {out_file}")

    # instrument sharpe
    raw_sharpe = calc_sharpe(df["forecast_x_return"])
    exec_sharpe = calc_sharpe(df["strategy_ret"])

    # turnover
    turnover, avg_traded, avg_pos = compute_turnover(df)
    print(
        f"{instr_name}: raw Sharpe={raw_sharpe:.3f}, exec Sharpe={exec_sharpe:.3f}, "
        f"turnover={turnover:.2f}, yearly lots={avg_traded:.1f}, avg|pos|={avg_pos:.1f}"
    )

    return df, raw_sharpe, exec_sharpe

# ==============================================================
# 4) LOAD INPUT FILES (WITH CORRECT COLUMN NAMES)
# ==============================================================

rx = pd.read_csv(RX_FILE)
ad = pd.read_csv(AD_FILE)

# your files have uppercase names: PX_CLOSE_1D, not px_close_1d
rx["Date"] = pd.to_datetime(rx["Date"], dayfirst=True, errors="coerce")
ad["Date"] = pd.to_datetime(ad["Date"], dayfirst=True, errors="coerce")

rx = rx.sort_values("Date").dropna(subset=["Date"])
ad = ad.sort_values("Date").dropna(subset=["Date"])

# ==============================================================
# 5) PDM FROM UNDERLYING PRICES (USING PX_CLOSE_1D)
# ==============================================================

merged = pd.merge(
    rx[["Date", "PX_CLOSE_1D"]],
    ad[["Date", "PX_CLOSE_1D"]],
    on="Date",
    how="inner",
    suffixes=("_rx", "_ad")
).dropna()

merged["ret_px_rx"] = merged["PX_CLOSE_1D_rx"].pct_change()
merged["ret_px_ad"] = merged["PX_CLOSE_1D_ad"].pct_change()

corr = merged[["ret_px_rx", "ret_px_ad"]].corr()
print("\nCorrelation matrix (underlying returns):")
print(corr)

w = np.array([WEIGHTS["RX1"], WEIGHTS["AD1"]])
pdm_raw = 1.0 / np.sqrt(w.T @ corr.values @ w)
# Carver cap
PDM = min(pdm_raw, 2.5)
print(f"\nRaw PDM: {pdm_raw:.4f} → Capped PDM (used): {PDM:.4f}")

# ==============================================================
# 6) RUN BOTH INSTRUMENTS
# ==============================================================

rx_df, rx_raw_sh, rx_exec_sh = run_instrument(rx, "RX1", WEIGHTS["RX1"], PDM, OUTPUT_DIR)
ad_df, ad_raw_sh, ad_exec_sh = run_instrument(ad, "AD1", WEIGHTS["AD1"], PDM, OUTPUT_DIR)

print(f"\nRX1 raw Sharpe (signal):     {rx_raw_sh:.3f}")
print(f"RX1 PnL Sharpe (executed):   {rx_exec_sh:.3f}")
print(f"AD1 raw Sharpe (signal):     {ad_raw_sh:.3f}")
print(f"AD1 PnL Sharpe (executed):   {ad_exec_sh:.3f}")

# ==============================================================
# 7) PORTFOLIO SHARPE (RAW + EXEC)
# ==============================================================

combo = pd.DataFrame(index=rx_df.index.union(ad_df.index)).sort_index()
combo["ret_rx_exec"] = rx_df["strategy_ret"]
combo["ret_ad_exec"] = ad_df["strategy_ret"]
combo["ret_port_exec"] = WEIGHTS["RX1"] * combo["ret_rx_exec"].fillna(0) + WEIGHTS["AD1"] * combo["ret_ad_exec"].fillna(0)

combo["ret_rx_raw"] = rx_df["forecast_x_return"]
combo["ret_ad_raw"] = ad_df["forecast_x_return"]
combo["ret_port_raw"] = WEIGHTS["RX1"] * combo["ret_rx_raw"].fillna(0) + WEIGHTS["AD1"] * combo["ret_ad_raw"].fillna(0)

port_sharpe_exec = calc_sharpe(combo["ret_port_exec"])
port_sharpe_raw = calc_sharpe(combo["ret_port_raw"])

print("\n=== Portfolio Sharpe Ratios ===")
print(f"Raw (signal-only):     {port_sharpe_raw:.3f}")
print(f"Executed (realized):   {port_sharpe_exec:.3f}")

# save portfolio too
portfolio_out = os.path.join(OUTPUT_DIR, "gpt_portfolio_rx_ad.csv")
combo.to_csv(portfolio_out)
print(f"\n✅ Saved portfolio to {portfolio_out}")



Correlation matrix (underlying returns):
           ret_px_rx  ret_px_ad
ret_px_rx   1.000000  -0.024554
ret_px_ad  -0.024554   1.000000

Raw PDM: 1.4319 → Capped PDM (used): 1.4319
✅ Saved RX1 to C:\Users\loci_\Desktop\trading_webapp\DATA\RX1_full_chain.csv
RX1: raw Sharpe=-0.567, exec Sharpe=-0.340, turnover=30.51, yearly lots=3870.8, avg|pos|=63.4
✅ Saved AD1 to C:\Users\loci_\Desktop\trading_webapp\DATA\AD1_full_chain.csv
AD1: raw Sharpe=-0.917, exec Sharpe=-1.096, turnover=37.33, yearly lots=9509.7, avg|pos|=127.4

RX1 raw Sharpe (signal):     -0.567
RX1 PnL Sharpe (executed):   -0.340
AD1 raw Sharpe (signal):     -0.917
AD1 PnL Sharpe (executed):   -1.096

=== Portfolio Sharpe Ratios ===
Raw (signal-only):     -1.074
Executed (realized):   -0.965

✅ Saved portfolio to C:\Users\loci_\Desktop\trading_webapp\DATA\gpt_portfolio_rx_ad.csv
